## Training your own openWakeWord models


**SmartCozmo copy of openWakeWord's simple training notebook**, patched so it runs on Colab as of 2026-10. It trained `models/hey_cozmo.onnx`. See docs/wake-word.md, "Training a custom wake word" section.

**Before running:**
1. **Runtime → Change runtime type**: Hardware accelerator **T4 GPU**, Runtime version **2025.07**. Newer runtimes are Python 3.12/3.13, which have no `piper-phonemize` / `torchvision==0.20.0` builds, so cell 1 fails.
2. Run the `python --version` cell. It must say **3.11**.
3. In cell 1, type the wake phrase using **dictionary words separated by spaces**, e.g. `hey cosmo`, not `hey cozmo` or `hey_cosmo`. A word that's not in the pronunciation dictionary (CMUdict), including any word joined with `_`, makes training download a helper model from an S3 bucket that now returns 403, and training crashes with `invalid load key, '<'`. Spelling doesn't matter to the model, only sound: "cosmo" sounds like "Cozmo".
4. **Runtime → Run all**. It takes about 1-1.5 h and downloads the model at the end.

**What was changed from upstream:** the AudioSet download (cell 2.1, the old `.tar` link returns 404), a memory fix to `train.py` and CPU-only training in cell 3 (otherwise training is killed at 75%), and only the `.onnx` file is produced (the TFLite conversion is broken and not needed).

**If training is killed** (`Killed`, `exit code: 137`): **Runtime → Restart session** (keeps the files; don't use "Disconnect and delete runtime"), then run only cell 3. Generated clips and features are reused.

Upstream: https://github.com/dscripka/openWakeWord#training-new-models. Models trained with this data are for non-commercial personal use only (see cell 2).


In [ ]:
# Must print Python 3.11.x - see the notes above if not.
!python --version

In [ ]:
# @title  { display-mode: "form" }
# @markdown # 1. Test Example Training Clip Generation
# @markdown Since openWakeWord models are trained on synthetic examples of your
# @markdown target wake word, it's a good idea to make sure that the examples
# @markdown sound correct. Type in your target wake word below, and run the
# @markdown cell to listen to it.
# @markdown
# @markdown Here are some tips that can help get the wake word to sound right:

# @markdown - Use dictionary words separated by spaces. If a word isn't
# @markdown pronounced the way you want, respell it with a similar-sounding
# @markdown dictionary word ("hey cozmo" --> "hey cosmo"). Don't use
# @markdown underscores: they make an out-of-dictionary word, and training
# @markdown then crashes (see the notes at the top).

# @markdown - Spell out numbers ("2" --> "two")

# @markdown - Avoid all punctuation except for "?" and "!", and remove unicode characters

import os
import sys
from IPython.display import Audio

if not os.path.exists("./piper-sample-generator"):
    !git clone https://github.com/rhasspy/piper-sample-generator
    !wget -O piper-sample-generator/models/en_US-libritts_r-medium.pt 'https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt'
    !cd piper-sample-generator && git checkout 213d4d5

    # Install system dependencies
    !pip install piper-tts piper-phonemize-cross
    !pip install webrtcvad
    # !pip install 'torch<=2.5' torchvision torchaudio
    !pip install torch==2.5.0 torchvision==0.20.0 torchaudio==2.5.0 --index-url https://download.pytorch.org/whl/cu121

target_word = 'hey cosmo' # @param {type:"string"}
# "cozmo" isn't in the pronunciation dictionary, "cosmo" is and sounds the
# same. To check a word (after cell 2 has installed `pronouncing`):
#   import pronouncing; print(pronouncing.phones_for_word("cosmo"))  # [] = not found

if "piper-sample-generator/" not in sys.path:
    sys.path.append("piper-sample-generator/")
from generate_samples import generate_samples

def text_to_speech(text):
    generate_samples(text = text,
                max_samples=1,
                length_scales=[1.1],
                noise_scales=[0.7], noise_scale_ws = [0.7],
                output_dir = './', batch_size=1, auto_reduce_batch_size=True,
                file_names=["test_generation.wav"]
                )

text_to_speech(target_word)

from IPython.display import Audio
Audio("test_generation.wav", autoplay=True)


In [ ]:
# @title  { display-mode: "form" }
# @markdown # 2. Download Data
# @markdown Training custom models requires downloading a wide variety of data
# @markdown that will help make the model perform well in real-world scenarios.
# @markdown This example notebook will download small samples of background noise,
# @markdown music, and Room Impulse Responses (to add echo). This will still produce
# @markdown a custom model that performs well, but if you are interested in adding even more,
# @markdown feel free to extend this notebook to download the full datasets and even add
# @markdown your own!
# @markdown
# @markdown Downloading this example data will usually take about 15 minutes.

# @markdown **Important note!** The data downloaded here has a mixture of different
# @markdown licenses and usage restrictions. As such, any custom models trained with this
# @markdown data should be considered as appropriate for **non-commercial** personal use only.

# ## Install all dependencies
# !pip install datasets
# !pip install scipy
# !pip install tqdm

import locale
def getpreferredencoding(do_setlocale = True):
    return "UTF-8"
locale.getpreferredencoding = getpreferredencoding

# install openwakeword (full installation to support training)
!git clone https://github.com/dscripka/openwakeword
!pip install -e ./openwakeword --no-deps
# !cd openwakeword

# install other dependencies
!pip install mutagen==1.47.0
!pip install torchinfo==1.8.0
!pip install torchmetrics==1.2.0
!pip install speechbrain==0.5.14
!pip install audiomentations==0.33.0
!pip install torch-audiomentations==0.11.0
!pip install acoustics==0.2.6
# !pip uninstall tensorflow -y
# !pip install tensorflow-cpu==2.8.1
# !pip install protobuf==3.20.3
# !pip install tensorflow_probability==0.16.0
# !pip install onnx_tf==1.10.0
!pip install onnxruntime==1.22.1 ai_edge_litert==1.4.0 onnxsim
!pip install onnx2tf
!pip install onnx==1.19.1
# !pip install ai_edge_litert==1.2.0
!pip install onnx_graphsurgeon
!pip install sng4onnx
!pip install pronouncing==0.2.0
!pip install datasets==2.14.6
!pip install deep-phonemizer==0.0.19

# Download required models (workaround for Colab)
import os
os.makedirs("./openwakeword/openwakeword/resources/models", exist_ok=True)
!wget https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.onnx -O ./openwakeword/openwakeword/resources/models/embedding_model.onnx
!wget https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.tflite -O ./openwakeword/openwakeword/resources/models/embedding_model.tflite
!wget https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.onnx -O ./openwakeword/openwakeword/resources/models/melspectrogram.onnx
!wget https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.tflite -O ./openwakeword/openwakeword/resources/models/melspectrogram.tflite

# Imports
import sys

if "piper-sample-generator/" not in sys.path:
    sys.path.append("piper-sample-generator/")
from generate_samples import generate_samples

import numpy as np
import torch
import sys
from pathlib import Path
import uuid
import yaml
import datasets
import scipy
from tqdm import tqdm

## Download all data

## Download MIR RIR data (takes about ~2 minutes)
output_dir = "./mit_rirs"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)
    !git lfs install
    !git clone https://huggingface.co/datasets/davidscripka/MIT_environmental_impulse_responses
    rir_dataset = datasets.Dataset.from_dict({"audio": [str(i) for i in Path("./MIT_environmental_impulse_responses/16khz").glob("*.wav")]}).cast_column("audio", datasets.Audio())
    # Save clips to 16-bit PCM wav files
    for row in tqdm(rir_dataset):
        name = row['audio']['path'].split('/')[-1]
        scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))

## Download noise and background audio (takes about ~3 minutes)

# Audioset Dataset (https://research.google.com/audioset/dataset/index.html)
# Download one part of the audioset .tar files, extract, and convert to 16khz
# For full-scale training, it's recommended to download the entire dataset from
# https://huggingface.co/datasets/agkphysics/AudioSet, and
# even potentially combine it with other background noise datasets (e.g., FSD50k, Freesound, etc.)

# Disabled: the dataset is now Parquet, so this .tar link returns 404.
# Cell 2.1 below downloads the same part (09) instead.
# if not os.path.exists("audioset"):
#     os.mkdir("audioset")

#     fname = "bal_train09.tar"
#     out_dir = f"audioset/{fname}"
#     link = "https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/data/" + fname
#     !wget -O {out_dir} {link}
#     !cd audioset && tar -xvf bal_train09.tar

#     output_dir = "./audioset_16k"
#     if not os.path.exists(output_dir):
#         os.mkdir(output_dir)

#     # Save clips to 16-bit PCM wav files
#     audioset_dataset = datasets.Dataset.from_dict({"audio": [str(i) for i in Path("audioset/audio").glob("**/*.flac")]})
#     audioset_dataset = audioset_dataset.cast_column("audio", datasets.Audio(sampling_rate=16000))
#     for row in tqdm(audioset_dataset):
#         name = row['audio']['path'].split('/')[-1].replace(".flac", ".wav")
#         scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))

# Free Music Archive dataset
# https://github.com/mdeff/fma

output_dir = "./fma"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)
    fma_dataset = datasets.load_dataset("rudraml/fma", name="small", split="train", streaming=True)
    fma_dataset = iter(fma_dataset.cast_column("audio", datasets.Audio(sampling_rate=16000)))

    # Save clips to 16-bit PCM wav files
    n_hours = 1  # use only 1 hour of clips for this example notebook, recommend increasing for full-scale training
    for i in tqdm(range(n_hours*3600//30)):  # this works because the FMA dataset is all 30 second clips
        row = next(fma_dataset)
        name = row['audio']['path'].split('/')[-1].replace(".mp3", ".wav")
        scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))
        i += 1
        if i == n_hours*3600//30:
            break

# Download pre-computed openWakeWord features for training and validation

# training set (~2,000 hours from the ACAV100M Dataset)
# See https://huggingface.co/datasets/davidscripka/openwakeword_features for more information
if not os.path.exists("./openwakeword_features_ACAV100M_2000_hrs_16bit.npy"):
    !wget https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy

# validation set for false positive rate estimation (~11 hours)
if not os.path.exists("validation_set_features.npy"):
    !wget https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/validation_set_features.npy

In [ ]:
# @title  { display-mode: "form" }
# @markdown # 2.1. Download Data
# @markdown Replacement for the broken bal_train09.tar download: the dataset is now Parquet.
import io, os
from math import gcd
import numpy as np
import soundfile as sf
import pyarrow.parquet as pq
from scipy.signal import resample_poly
from huggingface_hub import hf_hub_download

OUT_DIR = "audioset_16k"   # must match the background-audio folder the notebook uses
os.makedirs(OUT_DIR, exist_ok=True)

path = hf_hub_download(repo_id="agkphysics/AudioSet", repo_type="dataset",
                       filename="data/bal_train/09.parquet")
pf = pq.ParquetFile(path)
print(pf.schema_arrow)  # expect an 'audio' column containing 'bytes'

n_ok = n_bad = 0
first_err = None
for batch in pf.iter_batches(batch_size=64, columns=["audio"]):
    for item in batch.column("audio").to_pylist():
        try:
            data, sr = sf.read(io.BytesIO(item["bytes"]), dtype="float32")
            if data.ndim > 1:                      # stereo -> mono
                data = data.mean(axis=1)
            if sr != 16000:                        # openWakeWord expects 16 kHz
                g = gcd(sr, 16000)
                data = resample_poly(data, 16000 // g, sr // g)
            sf.write(os.path.join(OUT_DIR, f"as09_{n_ok:05d}.wav"),
                     np.clip(data, -1, 1), 16000, subtype="PCM_16")
            n_ok += 1
        except Exception as e:
            n_bad += 1
            first_err = first_err or repr(e)
print(f"wrote {n_ok} clips to {OUT_DIR}, skipped {n_bad}; first error: {first_err}")


In [ ]:
# @title  { display-mode: "form" }
# @markdown # 3. Train the Model
# @markdown Now that you have verified your target wake word and downloaded the data,
# @markdown the last step is to adjust the training paramaters (or keep
# @markdown the defaults below) and start the training!

# @markdown Each paramater controls a different aspect of training:
# @markdown - `number_of_examples` controls how many examples of your wakeword
# @markdown are generated. The default (1,000) usually produces a good model,
# @markdown but between 30,000 and 50,000 is often the best.

# @markdown - `number_of_training_steps` controls how long to train the model.
# @markdown Similar to the number of examples, the default (10,000) usually works well
# @markdown but training longer usually helps.

# @markdown - `false_activation_penalty` controls how strongly false activations
# @markdown are penalized during the training process. Higher values can make the model
# @markdown much less likely to activate when it shouldn't, but may also cause it
# @markdown to not activate when the wake word isn't spoken clearly and there is
# @markdown background noise.

# @markdown With the default values shown below,
# @markdown this takes about 30 - 60 minutes total on the normal CPU Colab runtime.
# @markdown If you want to train on more examples or train for longer,
# @markdown try changing the runtime type to a GPU to significantly speedup
# @markdown the example generating and model training.

# @markdown When the model finishes training, `<output_name>.onnx` is downloaded
# @markdown automatically (also in the file browser on the left). Put it in
# @markdown SmartCozmo's models/ folder and set WAKE_WORD_MODEL=<output_name>.

import os
import sys
import yaml

# After Runtime > Restart session, target_word from cell 1 is gone - take it
# from the config the previous run saved.
try:
    target_word
except NameError:
    target_word = yaml.safe_load(open("my_model.yaml"))["target_phrase"][0]

# Name of the downloaded file (the phrase's respelling doesn't matter to the
# model, so it can be named after the real wake word).
output_name = 'hey_cozmo' # @param {type:"string"}

# Load default YAML config file for training
config = yaml.load(open("openwakeword/examples/custom_model.yml", 'r').read(), yaml.Loader)

# Modify values in the config and save a new version
number_of_examples = 10000 # @param {type:"slider", min:100, max:50000, step:50}
number_of_training_steps = 15000  # @param {type:"slider", min:0, max:50000, step:100}
false_activation_penalty = 1500  # @param {type:"slider", min:100, max:5000, step:50}
config["target_phrase"] = [target_word]
config["model_name"] = config["target_phrase"][0].replace(" ", "_")
config["n_samples"] = number_of_examples
config["n_samples_val"] = max(500, number_of_examples//10)
config["steps"] = number_of_training_steps
config["target_accuracy"] = 0.5
config["target_recall"] = 0.25
config["output_dir"] = "./my_custom_model"
config["max_negative_weight"] = false_activation_penalty

config["background_paths"] = ['./audioset_16k', './fma']  # multiple background datasets are supported
config["false_positive_validation_data_path"] = "validation_set_features.npy"
config["feature_data_files"] = {"ACAV100M_sample": "openwakeword_features_ACAV100M_2000_hrs_16bit.npy"}

with open('my_model.yaml', 'w') as file:
    documents = yaml.dump(config, file)

# Generate clips
!{sys.executable} openwakeword/openwakeword/train.py --training_config my_model.yaml --generate_clips

# Step 2: Augment the generated clips

!{sys.executable} openwakeword/openwakeword/train.py --training_config my_model.yaml --augment_clips

# Step 3: Train model.
# Patch train.py first: it validates against the ~11 h false-positive set in
# one batch, which runs Colab out of memory (killed at 75% on GPU, at the
# start of stage 2 on CPU). Batches of 8192 give the same result - false
# positives are summed across batches. Safe to run more than once.
!sed -i 's/batch_size=len(X_val_fp_labels)/batch_size=8192/' openwakeword/openwakeword/train.py
!grep -c "batch_size=8192" openwakeword/openwakeword/train.py

# Train on CPU (CUDA_VISIBLE_DEVICES=-1): GPU validation runs out of GPU
# memory. Three stages (steps, then 2 x steps/10) - about 15 min at 15,000.
# The script ends with a TFLite conversion error (`No module named
# 'onnx_tf'`) after the .onnx is saved - harmless, TFLite isn't needed.
!CUDA_VISIBLE_DEVICES=-1 {sys.executable} openwakeword/openwakeword/train.py --training_config my_model.yaml --train_model

# Download the trained model
import shutil
from google.colab import files

model_path = f"my_custom_model/{config['model_name']}.onnx"
if not os.path.exists(model_path):
    raise RuntimeError(f"{model_path} wasn't created - check the training output above for "
                       "'Killed' (see the notes at the top) or the first Traceback.")
shutil.copy(model_path, f"{output_name}.onnx")
files.download(f"{output_name}.onnx")
